In [1]:
# ============================================================
# CELL 1: Load mBART-50 + LoRA + reproduce test split + sanity
# ============================================================
import os, glob, json
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
from peft import PeftModel

os.chdir('/kaggle/working')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

# ---------- Auto-detect model directory ----------
# User should adjust this if needed
MODEL_DIR = "/kaggle/input/datasets/siddhanttimalsina/mbart-model"  # ← user may need to change

# If not found, try to auto-detect
if not os.path.exists(MODEL_DIR):
    candidates = glob.glob('/kaggle/input/**/adapter_config.json', recursive=True)
    if candidates:
        MODEL_DIR = os.path.dirname(candidates[0])
    else:
        candidates = glob.glob('/kaggle/input/**/config.json', recursive=True)
        # filter out config.json that isn't mBART
        for c in candidates:
            with open(c, 'r') as f:
                try:
                    cfg = json.load(f)
                    if cfg.get('model_type') == 'mbart' or 'MBart' in str(cfg.get('architectures', [])):
                        MODEL_DIR = os.path.dirname(c)
                        break
                except:
                    pass

print(f"Model dir: {MODEL_DIR}")

# ---------- Detect LoRA vs merged ----------
is_lora = os.path.exists(os.path.join(MODEL_DIR, 'adapter_config.json'))
print(f"Is LoRA adapter: {is_lora}")

# ---------- Load tokenizer ----------
BASE_MODEL_NAME = "facebook/mbart-large-50-many-to-many-mmt"
try:
    tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_DIR)
    print("Tokenizer loaded from local dir")
except Exception as e:
    print(f"Local tokenizer failed ({e}), loading from base")
    tokenizer = MBart50TokenizerFast.from_pretrained(BASE_MODEL_NAME)

tokenizer.src_lang = "ne_NP"
tokenizer.tgt_lang = "ne_NP"
print(f"Tokenizer: {type(tokenizer).__name__}, vocab={tokenizer.vocab_size}")
print(f"src_lang={tokenizer.src_lang}, tgt_lang={tokenizer.tgt_lang}")
print(f"Nepali lang token id: {tokenizer.lang_code_to_id.get('ne_NP')}")

# ---------- Load model ----------
if is_lora:
    print("\nLoading base model + LoRA adapter...")
    base = MBartForConditionalGeneration.from_pretrained(BASE_MODEL_NAME, torch_dtype=torch.float16)
    model = PeftModel.from_pretrained(base, MODEL_DIR)
    model = model.merge_and_unload()  # merge LoRA into base for faster inference
else:
    print("\nLoading merged model...")
    model = MBartForConditionalGeneration.from_pretrained(MODEL_DIR, torch_dtype=torch.float16)

model = model.to(device).eval()
print(f"mBART-50 loaded: {model.num_parameters()/1e6:.1f}M params")

Device: cpu
Model dir: /kaggle/input/datasets/siddhanttimalsina/mbart-large-lora
Is LoRA adapter: False
Tokenizer loaded from local dir
Tokenizer: MBart50Tokenizer, vocab=250054
src_lang=ne_NP, tgt_lang=ne_NP
Nepali lang token id: 250018

Loading merged model...


Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

mBART-50 loaded: 610.9M params
